# Faz 0 — Hakem A/B tezgahı (Kaggle)

`kaggle_run_gemma_judge.ipynb` ile aynı altyapı; farkı **ne koştuğu**: envanter
batch'i değil, 125 sorgu × 3 çağrılık kontrollü bir A/B ölçümü.

| karşılaştırma | ne ölçer |
|---|---|
| `v1` ↔ `v4` | **asıl soru** — v4 parent'ta temkin kazanıyor mu |
| `v3` ↔ `v4` | v4, v3'ün `null`/subunit hasarından kaçınıyor mu |
| `v1` ↔ `v3` | 14 Ağustos ölçümü (35 fark) yeni oturumda tekrarlanıyor mu |

**Gürültü tabanı 14 Ağustos'ta ölçüldü: %0** (125 sorgu, iki bağımsız geçiş,
`raw_response` dahil bayt-denk). Bu yüzden tekrar koşusu YOK — tek koşu yeterli
ve gözlenen her fark gerçek.

## ⚠️ Zorunlu ön koşullar
1. **Accelerator = GPU T4 x2** (P100 DEĞİL — `sentence-transformers` sm_60'ta çöker).
2. **Internet açık** (telefon doğrulaması gerekiyor).
3. **Add Input** ile üç dataset ekli olmalı:
   - `mcangultekin/gemma4-e4b-ollama`
   - `mcangultekin/institution-catalog-canonical`
   - `mcangultekin/kaggle-judge-output`  ← ölçüm seti bundan **yeniden üretilir**

Ölçüm setini yüklemene gerek yok: `build_faz0_sample.py` onu `kaggle_judge_sonuc.csv`'den
hash tabanlı, sürümden bağımsız seçimle yeniden üretir. Yereldeki mühürle
karşılaştırılır — tutmazsa hücre durur.

## Tasarım notları
1. `resolve()` sorgu başına **bir kez**; tüm varyantlar **aynı aday havuzunu** görür.
2. Tekrarlar **ayrı geçişte** koşar — peş peşe tekrar Ollama KV-cache'ini yeniden
   kullanıyor (21,97 sn → 2,03 sn ölçüldü) ve çıktıyı yapay olarak aynı yapıyor.
3. `OLLAMA_NUM_PARALLEL=1` — paralellik sürekli batching yüzünden kendisi
   kararsızlık ekleyebilir; ölçtüğümüz şey tam da bu.



## 1) GPU kontrolü


In [ ]:
!nvidia-smi

import subprocess
gpu = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print('GPU:', gpu or '(YOK)')
if 'P100' in gpu:
    raise RuntimeError('P100 secili - Session options -> Accelerator -> GPU T4 x2')
if not gpu:
    raise RuntimeError('GPU yok - Session options -> Accelerator -> GPU T4 x2')


## 2) Dataset yolları


In [ ]:
import os

KAGGLE_USERNAME = 'mcangultekin'
MODEL_DS   = '/kaggle/input/datasets/mcangultekin/gemma4-e4b-ollama'
CATALOG_DS = '/kaggle/input/datasets/mcangultekin/institution-catalog-canonical'
JUDGE_OUT  = '/kaggle/input/datasets/mcangultekin/kaggle-judge-output/kaggle_judge_sonuc.csv'

for p, ad in ((MODEL_DS,'model'), (CATALOG_DS,'katalog')):
    assert os.path.isdir(p), f'{ad} dataset bulunamadi: {p}'
assert os.path.isfile(JUDGE_OUT), f'hakem ciktisi bulunamadi: {JUDGE_OUT}'
print('Uc dataset de hazir.')


## 3) Kod


In [ ]:
REPO_DIR = '/kaggle/working/institution_resolver_v3'
BRANCH   = 'feat/gate-asama1'

import os
if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} https://github.com/mcangultekin/institution_resolver_v3.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull
%cd {REPO_DIR}

# Tezgah bu commit'te geldi - eski bir klon varsa erken uyar
assert os.path.isfile('scripts/judge_ab.py'), 'judge_ab.py yok - git pull calisti mi?'


## 4) Katalog dosyaları


In [ ]:
import shutil, os
os.chdir(REPO_DIR)
os.makedirs('data/processed', exist_ok=True)
for fn in ('parent_canonical.jsonl', 'subunit_canonical.jsonl'):
    shutil.copy(f'{CATALOG_DS}/{fn}', f'data/processed/{fn}')
print('Katalog:', os.listdir('data/processed'))


## 5) Ölçüm setini yeniden üret

Seçim `random` değil **sha256 tabanlı** — Python sürümünden bağımsız.
Mühür yereldekiyle aynı çıkmalı, yoksa iki ortamda farklı set ölçerdik.


In [ ]:
YEREL_MUHUR = 'd34036434632c6dc'   # yerelde uretilen setin muhru

!python3 scripts/build_faz0_sample.py --source '{JUDGE_OUT}' --out data/eval/faz0_ornek_125.csv

import csv, hashlib
with open('data/eval/faz0_ornek_125.csv', encoding='utf-8') as f:
    qs = sorted(r['query'] for r in csv.DictReader(f))
muhur = hashlib.sha256('\n'.join(qs).encode()).hexdigest()[:16]
print('muhur:', muhur, '| yerel:', YEREL_MUHUR)
assert muhur == YEREL_MUHUR, 'ORNEKLEM FARKLI! Kaynak CSV ayni mi?'
print(f'{len(qs)} sorgu - yerel setle OZDES')


## 6) Elasticsearch


In [ ]:
%%bash
set -e
ES_VERSION=8.14.0
if [ ! -d /kaggle/working/es ]; then
  wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-${ES_VERSION}-linux-x86_64.tar.gz -O /kaggle/working/es.tar.gz
  mkdir -p /kaggle/working/es
  tar -xzf /kaggle/working/es.tar.gz -C /kaggle/working/es --strip-components=1
fi

grep -q '^discovery.type' /kaggle/working/es/config/elasticsearch.yml || cat >> /kaggle/working/es/config/elasticsearch.yml <<EOF
discovery.type: single-node
xpack.security.enabled: false
xpack.security.http.ssl.enabled: false
xpack.ml.enabled: false
EOF

cat > /kaggle/working/es/config/elasticsearch.policy <<'POLEOF'
grant {
  permission java.io.FilePermission "/sys/-", "read";
  permission java.io.FilePermission "/proc/-", "read";
};
POLEOF

sysctl -w vm.max_map_count=262144 || true
id -u esuser &>/dev/null || useradd -m esuser
chown -R esuser:esuser /kaggle/working/es

pkill -f 'org.elasticsearch.bootstrap.Elasticsearch' 2>/dev/null || true
sleep 1
sudo -u esuser env ES_JAVA_OPTS="-Xms4g -Xmx4g -Djava.security.policy=/kaggle/working/es/config/elasticsearch.policy" \
  setsid /kaggle/working/es/bin/elasticsearch < /dev/null > /kaggle/working/es/es.log 2>&1 &
disown
sleep 3


In [ ]:
import time, requests
for _ in range(60):
    try:
        r = requests.get('http://localhost:9200/_cluster/health', timeout=2)
        if r.status_code == 200:
            print('ES saglikli:', r.json()['status']); break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError('ES baslatilamadi - /kaggle/working/es/es.log')


## 7) Paketler


In [ ]:
import os
os.chdir(REPO_DIR)
!pip uninstall -y torchaudio torchvision 2>/dev/null || true
!pip install -q --force-reinstall -e '.[dev,embed,llm,api]'


## 8) Ollama + gemma4:e4b

`OLLAMA_MODELS` doğrudan salt-okunur dataset'e bakıyor — `ollama pull` yok.


In [ ]:
%%bash
set -e
apt-get update -qq && apt-get install -y -qq zstd
if ! command -v ollama &> /dev/null; then
  curl -fsSL https://ollama.com/download/ollama-linux-amd64.tar.zst | tar --zstd -x -C /usr
fi
ollama --version


In [ ]:
import os, subprocess, time, requests

os.environ['OLLAMA_MODELS'] = MODEL_DS
os.environ['OLLAMA_KEEP_ALIVE'] = '-1'
os.environ['OLLAMA_NUM_PARALLEL'] = '1'   # olcumun temizligi icin - bkz. baslik

subprocess.Popen(['ollama','serve'], stdout=open('/kaggle/working/ollama.log','w'),
                 stderr=subprocess.STDOUT, env=os.environ)

for _ in range(30):
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=2)
        if r.status_code == 200:
            names = [m['name'] for m in r.json().get('models', [])]
            print('Ollama saglikli:', names)
            assert any('gemma4' in n for n in names), 'gemma4:e4b listede yok'
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(1)
else:
    raise RuntimeError('Ollama baslamadi - /kaggle/working/ollama.log')


## 9) İndeksleme

Embedding önbelleği yok (dataset'te `embeddings.npz` bulunmuyor), 231k kayıt
sıfırdan kodlanır — ~20 dk sürer.


In [ ]:
import os
os.chdir(REPO_DIR)
!python3 -m institution_resolver_v3.cli.main setup-es
!python3 -m institution_resolver_v3.cli.main index --embeddings


## 10) Duman testi

Burada bakılacak şey **kararlar değil**, tezgahın üç garantisi.


In [ ]:
!python3 scripts/judge_ab.py run --variants v1,v4,v5-bagli --limit 3 --out /kaggle/working/ab_smoke.csv



In [ ]:
import csv
csv.field_size_limit(10_000_000)
rows = list(csv.DictReader(open('/kaggle/working/ab_smoke.csv', encoding='utf-8')))
ok = True
for q in dict.fromkeys(r['query'] for r in rows):
    g = [r for r in rows if r['query'] == q]
    havuz = len({r['candidates_json'] for r in g}) == 1
    h = {r['variant']: r['prompt_sha256'] for r in g}
    ucu_farkli = len(set(h.values())) == 3
    ok &= havuz and ucu_farkli
    print(f"{q[:34]:<36} havuz_tek={havuz}  uc_prompt_farkli={ucu_farkli}")
print()
for r in rows:
    print(f"{r['query'][:30]:<32} {r['variant']:<4} {r['judge_s']:>6}s  {r['parent_verdict']}")
assert ok, 'Tezgah garantileri saglanmadi - tam kosuya GECME'
print('\nGarantiler tamam.')


## 11) TAM KOŞU — 125 sorgu × 3 varyant

Tek geçiş: her sorgu için `v1`, `v3`, `v4` arka arkaya (zaman kayması üçüne eşit dağılsın).
375 çağrı, önceki koşudaki hıza göre (~5 sn/çağrı) kabaca **30-35 dakika**.

Kesinti olursa aynı hücreyi tekrar çalıştır — `--resume` kaldığı yerden devam eder.


In [ ]:
import os
OUT = '/kaggle/working/ab_faz0_v5.csv'
RESUME = '--resume' if os.path.exists(OUT) else ''
!python3 scripts/judge_ab.py run --variants v1,v4,v5-bagli --out {OUT} {RESUME}




## 12) Ölçümler


### 12a) ASIL SORU — `v1` ↔ `v4`

Yanlışlanabilir tahmin: v4, v1'e göre **daha az `auto_match`** vermeli (v3'ün 16
kazancı: doğru cevabın havuzda olmadığı sorgularda `no_match`), ama `null`/subunit
davranışını **bozmamalı** (v3'ün 18 kaybı tekrarlanmamalı).


In [ ]:
!python3 scripts/judge_ab.py diff --run {OUT} --a v1#0 --b v4#0 --out /kaggle/working/fark_v1_v4.csv


### 12b) `v3` ↔ `v4` — hasar giderildi mi

Fark, ağırlıklı olarak `null`/subunit vakalarında olmalı; parent kararları
büyük ölçüde AYNI kalmalı (ikisinde de kural blokları çıkarılmış durumda).


In [ ]:
!python3 scripts/judge_ab.py diff --run {OUT} --a v4#0 --b v5-bagli#0 --out /kaggle/working/fark_v4_v5.csv



### 12c) `v1` ↔ `v3` — 14 Ağustos ölçümü tekrarlanıyor mu

O koşuda 35/125 fark çıkmıştı. Benzer bir sayı ve profil çıkarsa bulgu sağlam;
çok farklıysa koşular arası kayma sandığımızdan büyük demektir.


In [ ]:
!python3 scripts/judge_ab.py diff --run {OUT} --a v1#0 --b v5-bagli#0 --out /kaggle/working/fark_v1_v5.csv



### 12d) Karar dağılımı ve süre


In [ ]:
import csv, collections, statistics as st
csv.field_size_limit(10_000_000)
rows = list(csv.DictReader(open(OUT, encoding='utf-8')))

print(f"{'varyant':<9} {'auto':>5} {'review':>7} {'ambig':>6} {'nomatch':>8} {'HATA':>5} {'prompt':>8} {'medyan sn':>10}")
for key in sorted({(r['variant'], r['repeat_idx']) for r in rows}):
    g = [r for r in rows if (r['variant'], r['repeat_idx']) == key]
    c  = collections.Counter(r['parent_verdict'] if r['status']=='ok' else 'HATA' for r in g)
    ch = st.median([int(r['prompt_chars']) for r in g if r['prompt_chars']])
    sn = st.median([float(r['judge_s']) for r in g if r['judge_s']])
    print(f"{key[0]}#{key[1]:<6} {c['auto_match']:>5} {c['review']:>7} {c['ambiguous']:>6} "
          f"{c['no_match']:>8} {c['HATA']:>5} {ch:>8.0f} {sn:>10.2f}")

print('\nHATA turleri:')
for k, v in collections.Counter(r['error'][:70] for r in rows if r['status']!='ok').most_common():
    print(f'  {v:>4}  {k}')


## 13) Sonuçları sakla

`/kaggle/working/` oturum bitince silinir. İki yol: sağ panelden **Download**,
ya da aşağıdaki hücreyle Kaggle Dataset'ine push (token `KAGGLE_API_TOKEN`
secret'ında tanımlıysa).


In [ ]:
import os, shutil
os.makedirs('/kaggle/working/faz0_sonuc', exist_ok=True)
for fn in ('ab_faz0_v5.csv','fark_v1_v4.csv','fark_v4_v5.csv','fark_v1_v5.csv'):
    pth = f'/kaggle/working/{fn}'
    if os.path.exists(pth):
        shutil.copy(pth, f'/kaggle/working/faz0_sonuc/{fn}')
        print(f'{fn}: {os.path.getsize(pth)/1e6:.1f} MB')
print('\nSag panel -> Output -> faz0_sonuc/ klasorunu indirin.')




---
## Sonucu nasıl okumalı

- **`judge_s` varyantlar arası doğrudan karşılaştırılamaz.** Prompt'lar ortak önek
  paylaştığı için aynı sorguda ikinci varyant kısmi önbellekten yararlanır.
  v3'ün hız kazancı ayrı ölçüm ister; **kararlar bundan etkilenmez.**
- **Hata sayısı tek başına yanıltıcı.** Bir varyantta hata azalıp yanlışlar sessizce
  `auto_match` içine dağılabilir — fark dosyalarındaki değişen kararlar tek tek
  incelenmeden sonuç çıkarma.
- **12a olmadan 12b/12c okunmaz.** Gürültü tabanından küçük fark = bulgu yok.
